Importing the Dependencies

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn import metrics

Data Collection and Processing

In [4]:
# loading the data from csv file to pandas dataframe
car_dataset = pd.read_csv('/content/car price data set.csv.csv')

In [5]:
# inspecting the first 5 rows of the dataframe
car_dataset.head()

,Unnamed: 0,make,model,city,year,mileage,Engine_displacement,Battery,Price_Rs
0,0,Suzuki,Alto,Lahore,2019,45744,660.0,NaN,3550000
1,1,Suzuki,Wagon,Lahore,2019,17583,660.0,NaN,3850000
2,2,Suzuki,Wagon,Lahore,2019,64085,660.0,NaN,3890000
3,3,Suzuki,Wagon,Lahore,2019,71281,660.0,NaN,3990000
4,4,Toyota,Aqua,Lahore,2020,19950,1500.0,NaN,3990000


In [6]:
# checking the number of rows and columns
car_dataset.shape

(86120, 9)

In [7]:
# getting some information about the dataset
car_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86120 entries, 0 to 86119
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Unnamed: 0           86120 non-null  int64  
 1   make                 86120 non-null  object 
 2   model                86120 non-null  object 
 3   city                 86120 non-null  object 
 4   year                 86120 non-null  int64  
 5   mileage              86120 non-null  int64  
 6   Engine_displacement  85931 non-null  float64
 7   Battery              189 non-null    float64
 8   Price_Rs             86120 non-null  int64  
dtypes: float64(2), int64(4), object(3)
memory usage: 5.9+ MB


In [8]:
# checking the number of missing values
car_dataset.isnull().sum()

,0
Unnamed: 0,0
make,0
model,0
city,0
year,0
mileage,0
Engine_displacement,189
Battery,85931
Price_Rs,0


In [9]:
car_dataset.head()

,Unnamed: 0,make,model,city,year,mileage,Engine_displacement,Battery,Price_Rs
0,0,Suzuki,Alto,Lahore,2019,45744,660.0,NaN,3550000
1,1,Suzuki,Wagon,Lahore,2019,17583,660.0,NaN,3850000
2,2,Suzuki,Wagon,Lahore,2019,64085,660.0,NaN,3890000
3,3,Suzuki,Wagon,Lahore,2019,71281,660.0,NaN,3990000
4,4,Toyota,Aqua,Lahore,2020,19950,1500.0,NaN,3990000


Splitting the data and Target

In [35]:
Q1 = car_dataset['Price_Rs'].quantile(0.25)
Q3 = car_dataset['Price_Rs'].quantile(0.75)
IQR = Q3 - Q1
upper_limit = Q3 + 1.5 * IQR
car_dataset = car_dataset[car_dataset['Price_Rs'] < upper_limit]
print("Outliers removed")

Outliers removed


In [40]:
Y = car_dataset['Price_Rs']
X = car_dataset.drop(['Price_Rs','Unnamed: 0','Battery'], axis=1)
X = X.dropna()
Y = Y[X.index]
X = pd.get_dummies(X, columns=['make', 'model', 'city'], drop_first=True)

print("X shape:", X.shape)
print("NaN check:", X.isnull().sum().sum())  # 0

X shape: (79158, 781)
NaN check: 0


In [41]:
print(X)

       year  mileage  Engine_displacement  make_Alfa  make_Audi  make_Austin  \
0      2019    45744                660.0      False      False        False   
1      2019    17583                660.0      False      False        False   
2      2019    64085                660.0      False      False        False   
3      2019    71281                660.0      False      False        False   
4      2020    19950               1500.0      False      False        False   
...     ...      ...                  ...        ...        ...          ...   
86115  2011    60000                800.0      False      False        False   
86116  2009   300000               2700.0      False      False        False   
86117  2017    44740                800.0      False      False        False   
86118  2010    94890               1000.0      False      False        False   
86119  2008    60800               1000.0      False      False        False   

       make_BAIC  make_BMW  make_Bugatt

Splitting Training and Test data

In [42]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state=42)

Model Training

1. Linear Regression

In [43]:
# loading the linear regression mode
from sklearn.linear_model import LinearRegression
lin_reg_model = LinearRegression()
lin_reg_model.fit(X_train, Y_train)

LinearRegression()

In [52]:
import numpy as np

car_dataset = car_dataset[car_dataset['Price_Rs'] > 0]
car_dataset = car_dataset.dropna(subset=['Price_Rs'])

X = car_dataset.drop(['Price_Rs', 'Unnamed: 0', 'Battery'], axis=1)
Y = car_dataset['Price_Rs']
Y_log = np.log1p(Y)

X = X.dropna()
Y_log = Y_log[X.index]

X = pd.get_dummies(X, columns=['make', 'model', 'city'], drop_first=True)

X_train, X_test, Y_train_log, Y_test_log = train_test_split(X, Y_log, test_size=0.2, random_state=42)

lin_reg_model = LinearRegression()
lin_reg_model.fit(X_train, Y_train_log)

print("Model train ho gaya ✅")

test_pred_log = lin_reg_model.predict(X_test)
test_pred = np.expm1(test_pred_log)
Y_test = np.expm1(Y_test_log)

from sklearn.metrics import r2_score
print("Log Space R²:", lin_reg_model.score(X_test, Y_test_log))
print("Real Price R²:", r2_score(Y_test, test_pred))
print("Min predicted price:", test_pred.min())

Model train ho gaya ✅
Log Space R²: 0.8940374406695715
Real Price R²: 0.8147214968290503
Min predicted price: 109893.19734162891


Model Evaluation

In [53]:
# prediction on Training data
training_data_prediction = lin_reg_model.predict(X_train)

Visualize the actual prices and Predicted prices

In [70]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

# R² Score Rs mein
r2 = r2_score(Y_test, test_pred)
print(f"Real Price R² Score: {r2:.4f}")

mae = mean_absolute_error(Y_test, test_pred)
print(f"Mean Absolute Error: Rs {mae:,.0f}")

rmse = np.sqrt(mean_squared_error(Y_test, test_pred))
print(f"Root Mean Squared Error: Rs {rmse:,.0f}")

# Accuracy % mein
print(f"Model Accuracy: {r2*100:.2f}%")

Real Price R² Score: 0.7461
Mean Absolute Error: Rs 792,763
Root Mean Squared Error: Rs 2,980,571
Model Accuracy: 74.61%
